<a href="https://colab.research.google.com/github/navikram03/data-cleaning/blob/main/test_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BRONZE

In [ ]:
%pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv("/content/test.csv", inferSchema=True, header=True)

df.columns

['Rank',
 'Peak',
 'All Time Peak',
 'Actual\xa0gross',
 'Adjusted\xa0gross (in 2022 dollars)',
 'Artist',
 'Tour title',
 'Year(s)',
 'Shows',
 'Average gross',
 'Ref.']

SILVER

In [ ]:
df_dict_names = {
    "Rank":"rank",
    "Peak":"peak",
    "All Time Peak":"all_time_peak",
    "Actual gross":"actual_gross",
    "Adjusted gross (in 2022 dollars)":"adjusted_gross",
    "Artist":"artist",
    "Tour title":"tour_title",
    "Year(s)":"year",
    "Shows":"shows",
    "Average gross":"average_gross",
    "Ref.":"reference"
}

df = df.toDF(*[column.replace('\xa0', ' ').strip() for column in df.columns])

for each in df.columns:
  new_name = df_dict_names[each]

  df = df.withColumnRenamed(each, new_name)
  if new_name in ["peak","all_time_peak"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*\]','' )).cast("int"))
  elif new_name in ["actual_gross", "adjusted_gross", "average_gross"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*\]','' )))
    df = df.withColumn(new_name, (regexp_replace(col(new_name),'[$,]','' )).cast("long"))
  elif new_name in ["artist", "tour_title"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*?]','' )))
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'[†‡\*]','' )).cast("string"))
  elif new_name == "year":
    df_clean = df.withColumn("year_clean", regexp_replace(col("year"), "[–—]", "-"))

    # 2. Split string on hyphen into an array column
    df_split = df_clean.withColumn("year_array", split(col("year_clean"), "-"))

    # 3. Extract 1st element for start_year and 2nd element (or fallback to 1st) for end_year
    df_final = df_split \
        .withColumn("start_year", element_at(col("year_array"), 1).cast("int")) \
        .withColumn("end_year", coalesce(element_at(col("year_array"), 2), element_at(col("year_array"), 1)).cast("int")) \
        # .drop("year_clean", "year_array")


In [ ]:
# df = df.fillna(0,subset=["peak","all_time_peak"])
df_final.show()

{"ts": "2026-08-19 10:03:00.482", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[INVALID_ARRAY_INDEX_IN_ELEMENT_AT] The index 2 is out of bounds. The array has 1 elements. Use `try_element_at` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "element_at", "errorClass": "INVALID_ARRAY_INDEX_IN_ELEMENT_AT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o5117.showString.\n: org.apache.spark.SparkArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX_IN_ELEMENT_AT] The index 2 is out of bounds. The array has 1 elements. Use `try_element_at` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003\n== DataFrame ==\n\"element_at\" was called from\njava.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)\n\n\tat org.apache.

ArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX_IN_ELEMENT_AT] The index 2 is out of bounds. The array has 1 elements. Use `try_element_at` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"element_at" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
